# ACC/HPC Decoding, GC, and RSA Subject-Level Correlations

Eleven subject-level Spearman (primary) / Pearson (secondary) correlations:

1. **A** — ACC RPE reinstatement vs ACC cue hit decoding (HFA)
2. **B** — ACC cue hit decoding (HFA) vs HPC→ACC theta GC (`gc_net_z`, cue / positive RPE / 0–1 s)
3. **C** — HPC theta hit decoding vs ACC→HPC hit RSA (significant cluster mean)
4. **D** — HPC theta hit decoding vs HPC→ACC theta GC (cue / positive RPE / 0–1 s)
5. **E** — ACC cue hit decoding (HFA) vs ACC→HPC hit RSA (significant cluster mean)
6. **F** — ACC RPE reinstatement (reactivation) vs HPC theta hit decoding (subjects with ACC + HPC)
7. **G** — HPC→ACC theta GC (`gc_net_z`, cue / positive RPE / 0–1 s) vs ACC→HPC hit RSA (significant cluster mean)
8. **H** — ACC↔HPC multivariate MI (cluster effect) vs ACC RPE reinstatement, ACC hit decoding, HPC theta hit decoding, and HPC→ACC GC
9. **I** — ACC cue hit decoding (HFA) vs HPC theta hit decoding
10. **J** — ACC RPE reinstatement vs subject-level `hits ~ zrpe` OLS beta (Day1 memory)
11. **K** — ACC RPE crossdecoding (reinstatement) vs ACC→HPC hit RSA (significant cluster mean)

In [ ]:
import os
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from mne.stats import permutation_cluster_1samp_test
from scipy.sparse import csr_matrix, eye, kron
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf

from plot_config import set_plot_style

set_plot_style()

In [ ]:
# ---- Config ----
reinstatement_metric = 'MeanAccuracy'
corr_type = 'spearman'
run_secondary_pearson = True
label_points = True

acc_roi = 'ACC'
gc_metric = 'gc_net_z'
freq_band_label = 'theta'
gc_roi_source = 'HPC'
gc_roi_target = 'ACC'

rsa_feedback_roi = 'ACC'
rsa_cue_roi = 'HPC'
hit_memory_label = 'remembered'
rsa_metric_name = 'HitRSA_cluster_mean'
rsa_metric_col = f'ACC_HPC_{rsa_metric_name}'

cluster_alpha = 0.05
cluster_n_permutations = 1000
cluster_seed = 42
cluster_tail = 1

time_windows: Dict[str, Tuple[float, float]] = {
    'cue_full': (0.0, 1.0),
}
time_window_labels: Dict[str, str] = {
    'cue_full': 'whole cue (0–1 s)',
}
gc_conditions: List[Tuple[str, str]] = [('cue', 'pos')]

acc_reinstatement_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/'
    'ACC_rpe_category_HFA_subject_reinstatement.csv'
)
acc_hit_decoding_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'ACC_memory_outcome_hit_miss_hfa_subject_reinstatement.csv'
)
hpc_theta_decoding_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'HPC_memory_outcome_hit_miss_theta_subject_reinstatement.csv'
)
gc_merged_csv = (
    '/Volumes/T7 Shield/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvgc_rpe/'
    'notebook_runs/cue_on_Day1_t0p0-1p0_sf50_4-8Hz_lag10_roie86d0eb437/merged_gc_rpe.csv'
)
rsa_group_results_pkl = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/RSA/crossregion/'
    'group_results_ACC_to_HPC.pkl'
)
acc_hpc_mi_csv = (
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/decoding_cue/hit_miss/'
    'ACC_HPC_memory_outcome_hit_miss_hfa_subject_mi_cluster_effect.csv'
)
acc_hpc_mi_metadata_csv = acc_hpc_mi_csv.replace(
    '_subject_mi_cluster_effect.csv',
    '_subject_mi_cluster_effect_metadata.csv',
)
behavior_load_dir = '/Volumes/T7 Shield/work/qasims01/MemoryBanditData/EMU'
behavior_combined_csv = f'{behavior_load_dir}/full_df_RWH.csv'
zrpe_hits_metric_col = 'zrpe_hits_beta'
min_trials_zrpe_hits_ols = 10
behavior_day1_condition = 'Day1'

analysis_prefix = 'acc_hpc_decoding_gc_rsa_correlations'
analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/acc_hpc_decoding_gc_rsa_correlations'
)
figure_output_root = Path(
    '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
)
figures_output_root = analysis_output_root / 'figures'
analysis_output_root.mkdir(parents=True, exist_ok=True)
figures_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

acc_metric_col = f'{acc_roi}_{reinstatement_metric}'
acc_hit_metric_col = 'ACC_hit_MeanAccuracy'
hpc_theta_metric_col = 'HPC_theta_MeanAccuracy'
mi_metric_col = 'ACC_HPC_MI_MeanAccuracy'

paths: Dict[str, Path] = {
    'summary_txt': analysis_output_root / f'{analysis_prefix}_summary.txt',
    'cluster_metadata_csv': analysis_output_root / f'{analysis_prefix}_rsa_cluster_metadata.csv',
    'a_merged_csv': analysis_output_root / f'{analysis_prefix}_A_acc_reinstatement_vs_acc_hit_merged.csv',
    'a_results_csv': analysis_output_root / f'{analysis_prefix}_A_acc_reinstatement_vs_acc_hit_results.csv',
    'a_scatter_pdf': figures_output_root / 'acc_reinstatement_vs_acc_hit_decoding_scatter.pdf',
    'b_merged_csv': analysis_output_root / f'{analysis_prefix}_B_acc_hit_vs_gc_merged.csv',
    'b_results_csv': analysis_output_root / f'{analysis_prefix}_B_acc_hit_vs_gc_results.csv',
    'b_scatter_pdf': figures_output_root / 'acc_hit_decoding_vs_hpc_acc_gc_pos_cue_full_scatter.pdf',
    'c_merged_csv': analysis_output_root / f'{analysis_prefix}_C_hpc_theta_vs_hit_rsa_merged.csv',
    'c_results_csv': analysis_output_root / f'{analysis_prefix}_C_hpc_theta_vs_hit_rsa_results.csv',
    'c_scatter_pdf': figures_output_root / 'hpc_theta_vs_acc_hpc_hit_rsa_scatter.pdf',
    'd_merged_csv': analysis_output_root / f'{analysis_prefix}_D_hpc_theta_vs_gc_merged.csv',
    'd_results_csv': analysis_output_root / f'{analysis_prefix}_D_hpc_theta_vs_gc_results.csv',
    'd_scatter_pdf': figures_output_root / 'hpc_theta_vs_hpc_acc_gc_pos_cue_full_scatter.pdf',
    'e_merged_csv': analysis_output_root / f'{analysis_prefix}_E_acc_hit_vs_hit_rsa_merged.csv',
    'e_results_csv': analysis_output_root / f'{analysis_prefix}_E_acc_hit_vs_hit_rsa_results.csv',
    'e_scatter_pdf': figures_output_root / 'acc_hit_decoding_vs_acc_hpc_hit_rsa_scatter.pdf',
    'f_merged_csv': analysis_output_root / f'{analysis_prefix}_F_acc_reinstatement_vs_hpc_theta_merged.csv',
    'f_results_csv': analysis_output_root / f'{analysis_prefix}_F_acc_reinstatement_vs_hpc_theta_results.csv',
    'f_scatter_pdf': figures_output_root / 'acc_reinstatement_vs_hpc_theta_hit_decoding_scatter.pdf',
    'g_merged_csv': analysis_output_root / f'{analysis_prefix}_G_gc_vs_hit_rsa_merged.csv',
    'g_results_csv': analysis_output_root / f'{analysis_prefix}_G_gc_vs_hit_rsa_results.csv',
    'g_scatter_pdf': figures_output_root / 'hpc_acc_gc_pos_vs_acc_hpc_hit_rsa_scatter.pdf',
    'h_merged_csv': analysis_output_root / f'{analysis_prefix}_H_mi_vs_decoding_gc_merged.csv',
    'h_results_csv': analysis_output_root / f'{analysis_prefix}_H_mi_vs_decoding_gc_results.csv',
    'h_scatter_pdf': figures_output_root / 'acc_hpc_mi_cluster_vs_decoding_gc_scatter.pdf',
    'i_merged_csv': analysis_output_root / f'{analysis_prefix}_I_acc_hit_vs_hpc_theta_merged.csv',
    'i_results_csv': analysis_output_root / f'{analysis_prefix}_I_acc_hit_vs_hpc_theta_results.csv',
    'i_scatter_pdf': figures_output_root / 'acc_hit_decoding_vs_hpc_theta_hit_decoding_scatter.pdf',
    'j_merged_csv': analysis_output_root / f'{analysis_prefix}_J_acc_reinstatement_vs_zrpe_hits_ols_merged.csv',
    'j_results_csv': analysis_output_root / f'{analysis_prefix}_J_acc_reinstatement_vs_zrpe_hits_ols_results.csv',
    'j_scatter_pdf': figures_output_root / 'acc_reinstatement_vs_zrpe_hits_ols_scatter.pdf',
    'k_merged_csv': analysis_output_root / f'{analysis_prefix}_K_acc_crossdecoding_vs_hit_rsa_merged.csv',
    'k_results_csv': analysis_output_root / f'{analysis_prefix}_K_acc_crossdecoding_vs_hit_rsa_results.csv',
    'k_scatter_pdf': figures_output_root / 'acc_crossdecoding_vs_acc_hpc_hit_rsa_scatter.pdf',
    'behavior_zrpe_hits_csv': analysis_output_root / f'{analysis_prefix}_behavior_zrpe_hits_ols.csv',
}

manuscript_figure_paths: Dict[str, Path] = {
    'a_scatter_pdf': figure_output_root / 'acc_reinstatement_vs_acc_hit_decoding_scatter.pdf',
    'b_scatter_pdf': figure_output_root / 'acc_hit_decoding_vs_hpc_acc_gc_pos_cue_full_scatter.pdf',
    'c_scatter_pdf': figure_output_root / 'hpc_theta_vs_acc_hpc_hit_rsa_scatter.pdf',
    'd_scatter_pdf': figure_output_root / 'hpc_theta_vs_hpc_acc_gc_pos_cue_full_scatter.pdf',
    'e_scatter_pdf': figure_output_root / 'acc_hit_decoding_vs_acc_hpc_hit_rsa_scatter.pdf',
    'f_scatter_pdf': figure_output_root / 'acc_reinstatement_vs_hpc_theta_hit_decoding_scatter.pdf',
    'g_scatter_pdf': figure_output_root / 'hpc_acc_gc_pos_vs_acc_hpc_hit_rsa_scatter.pdf',
    'h_scatter_pdf': figure_output_root / 'acc_hpc_mi_cluster_vs_decoding_gc_scatter.pdf',
    'i_scatter_pdf': figure_output_root / 'acc_hit_decoding_vs_hpc_theta_hit_decoding_scatter.pdf',
    'j_scatter_pdf': figure_output_root / 'acc_reinstatement_vs_zrpe_hits_ols_scatter.pdf',
    'k_scatter_pdf': figure_output_root / 'acc_crossdecoding_vs_acc_hpc_hit_rsa_scatter.pdf',
}

analysis_a_predictor_order = [acc_hit_metric_col]
analysis_a_predictor_labels = {acc_hit_metric_col: 'ACC HFA hit decoding mean accuracy'}

analysis_c_predictor_order = [rsa_metric_col]
analysis_c_predictor_labels = {rsa_metric_col: 'ACC→HPC hit RSA (cluster mean)'}

analysis_e_predictor_order = [rsa_metric_col]
analysis_e_predictor_labels = {rsa_metric_col: 'ACC→HPC hit RSA (cluster mean)'}

analysis_f_predictor_order = [acc_metric_col]
analysis_f_predictor_labels = {acc_metric_col: 'ACC RPE reinstatement (reactivation)'}

analysis_g_predictor_order = [gc_metric]
analysis_g_predictor_labels = {gc_metric: 'HPC→ACC theta GC (pos, cue 0–1 s)'}

analysis_h_predictor_order = [
    acc_metric_col,
    acc_hit_metric_col,
    hpc_theta_metric_col,
    gc_metric,
]
analysis_h_predictor_labels = {
    acc_metric_col: 'ACC RPE reinstatement (reactivation)',
    acc_hit_metric_col: 'ACC HFA hit decoding',
    hpc_theta_metric_col: 'HPC theta hit decoding',
    gc_metric: 'HPC→ACC theta GC (pos, cue 0–1 s)',
}

analysis_i_predictor_order = [acc_hit_metric_col]
analysis_i_predictor_labels = {acc_hit_metric_col: 'ACC HFA hit decoding'}

analysis_j_predictor_order = [zrpe_hits_metric_col]
analysis_j_predictor_labels = {
    zrpe_hits_metric_col: 'zRPE effect on hits (per-subject OLS)',
}

analysis_k_predictor_order = [acc_metric_col]
analysis_k_predictor_labels = {
    acc_metric_col: 'ACC RPE crossdecoding (reinstatement)',
}

acc_hit_target_label = 'ACC cue hit decoding (HFA)'
hpc_target_label = 'HPC theta hit decoding'

print('ACC reinstatement CSV:', acc_reinstatement_csv)
print('ACC hit decoding CSV:', acc_hit_decoding_csv)
print('HPC theta decoding CSV:', hpc_theta_decoding_csv)
print('GC merged CSV:', gc_merged_csv)
print('RSA group PKL:', rsa_group_results_pkl)
print('ACC-HPC MI CSV:', acc_hpc_mi_csv)
print('Behavior combined CSV:', behavior_combined_csv)
print('Analysis output root:', analysis_output_root)
print('Figures output root:', figures_output_root)
print('Manuscript figure root:', figure_output_root)

In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}')


def _gelman_standardize(values: pd.Series) -> pd.Series:
    arr = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    mu = np.nanmean(arr)
    sd = np.nanstd(arr)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=values.index)
    return pd.Series((arr - mu) / (2.0 * sd), index=values.index)


def build_zrpe_hits_ols_table(
    combined_csv: str,
    min_trials: int = 10,
    condition: str = 'Day1',
) -> pd.DataFrame:
    if not os.path.isfile(combined_csv):
        raise FileNotFoundError(f'Behavior combined CSV not found: {combined_csv}')

    raw = pd.read_csv(combined_csv)
    required = ['participant', 'hits', 'rpe', 'condition']
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise ValueError(f'combined_df missing columns: {missing}')

    df = raw.loc[raw['condition'] == condition].copy()
    df['hits'] = df['hits'].astype(float)
    df['rpe'] = pd.to_numeric(df['rpe'], errors='coerce')
    df = df.dropna(subset=['hits', 'rpe', 'participant']).reset_index(drop=True)

    rows = []
    skipped = []
    for subj in sorted(df['participant'].dropna().unique()):
        subj_df = df[df['participant'] == subj].copy()
        if len(subj_df) < min_trials:
            skipped.append((subj, len(subj_df), f'fewer than {min_trials} trials'))
            continue
        subj_df['zrpe'] = _gelman_standardize(subj_df['rpe'])
        subj_df = subj_df.dropna(subset=['hits', 'zrpe']).reset_index(drop=True)
        if len(subj_df) < min_trials:
            skipped.append((subj, len(subj_df), 'insufficient valid trials after z-score'))
            continue
        if subj_df['zrpe'].nunique() < 2:
            skipped.append((subj, len(subj_df), 'zrpe has no variance'))
            continue
        fit = smf.ols('hits ~ zrpe', data=subj_df).fit()
        rows.append({
            'Subject': subj,
            zrpe_hits_metric_col: float(fit.params['zrpe']),
            'zrpe_hits_intercept': float(fit.params['Intercept']),
            'zrpe_hits_r_squared': float(fit.rsquared),
            'n_trials': int(len(subj_df)),
        })

    if skipped:
        print('Skipped subjects in hits ~ zrpe OLS:')
        for subj, n, reason in skipped:
            print(f'  {subj}: n={n}, {reason}')

    if len(rows) == 0:
        raise ValueError('No subjects passed hits ~ zrpe OLS criteria.')

    return pd.DataFrame(rows).sort_values('Subject').reset_index(drop=True)


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def _extract_corr(res) -> Tuple[float, float]:
    if hasattr(res, 'statistic') and hasattr(res, 'pvalue'):
        return float(res.statistic), float(res.pvalue)
    return float(res[0]), float(res[1])


def run_correlation_analysis(
    x: np.ndarray,
    y: np.ndarray,
    corr_type: str = 'spearman',
    run_secondary_pearson: bool = True,
) -> Dict[str, float]:
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    n = int(len(x_valid))
    if n < 3:
        raise ValueError(f'Need at least 3 valid observations; got {n}.')

    rho_s, p_s = _extract_corr(spearmanr(x_valid, y_valid, nan_policy='omit'))
    if run_secondary_pearson or corr_type.lower() == 'pearson':
        r_p, p_p = _extract_corr(pearsonr(x_valid, y_valid))
    else:
        r_p, p_p = np.nan, np.nan

    corr_type_norm = corr_type.lower()
    if corr_type_norm == 'spearman':
        primary_r, primary_p = rho_s, p_s
    elif corr_type_norm == 'pearson':
        primary_r, primary_p = r_p, p_p
    else:
        raise ValueError(f"Unsupported corr_type: {corr_type}. Use 'spearman' or 'pearson'.")

    return {
        'n': n,
        'spearman_rho': rho_s,
        'spearman_p': p_s,
        'pearson_r': r_p,
        'pearson_p': p_p,
        'primary_corr_type': corr_type_norm,
        'primary_r': primary_r,
        'primary_p': primary_p,
    }


def _fit_and_plot_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    if len(x) < 2 or len(np.unique(x)) < 2:
        return
    coeff = np.polyfit(x, y, 1)
    xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    yfit = np.polyval(coeff, xfit)
    ax.plot(xfit, yfit, color='#222222', linewidth=1.2)


def _save_and_show_figure(fig, output_pdf: Path, mirror_pdf: Path = None) -> None:
    output_pdf = Path(output_pdf)
    output_pdf.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_pdf, dpi=300, bbox_inches='tight')
    png_path = output_pdf.with_suffix('.png')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    if mirror_pdf is not None:
        mirror_pdf = Path(mirror_pdf)
        mirror_pdf.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(mirror_pdf, dpi=300, bbox_inches='tight')
    display(fig)
    plt.close(fig)


def plot_subject_covariate_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    predictor_order: List[str],
    predictor_labels: Dict[str, str],
    output_pdf: Path,
    suptitle: str,
    label_points: bool = True,
    mirror_pdf: Path = None,
) -> None:
    n_panels = len(predictor_order)
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    sns.set_style('white')

    for ax, predictor in zip(axes, predictor_order):
        panel_df = merged_df[np.isfinite(merged_df[predictor]) & np.isfinite(merged_df[target_col])].copy()
        ax.set_title(predictor_labels[predictor])
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[predictor].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)
        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, predictor), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        rr = results_df[results_df['predictor'] == predictor].iloc[0]
        ann = (
            f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, "
            f"p={rr['primary_p']:.3g}, n={int(rr['n'])}"
        )
        ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)
        ax.set_xlabel(predictor_labels[predictor])
        ax.set_ylabel(target_col)
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    for ax in axes[n_panels:]:
        ax.axis('off')

    fig.suptitle(suptitle, fontsize=14)
    _save_and_show_figure(fig, output_pdf, mirror_pdf=mirror_pdf)


def average_gc_in_time_window(
    gc_df: pd.DataFrame,
    tmin: float,
    tmax: float,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
) -> pd.DataFrame:
    m = (
        (gc_df['time'] >= tmin)
        & (gc_df['time'] <= tmax)
        & (gc_df['roi_source'] == gc_roi_source)
        & (gc_df['roi_target'] == gc_roi_target)
    )
    sub = gc_df.loc[m].copy()
    if len(sub) == 0:
        raise ValueError(
            f'No GC rows for {gc_roi_source}->{gc_roi_target} in time [{tmin}, {tmax}].'
        )
    out = (
        sub.groupby(['participant', 'rpe_cat'], as_index=False, dropna=False)
        .agg(**{gc_metric: (gc_metric, 'mean')})
    )
    out = out.rename(columns={'participant': 'Subject'})
    out['task_condition'] = 'cue'
    return out


def plot_roi_condition_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    target_label: str,
    gc_col: str,
    conditions: List[Tuple[str, str]],
    output_pdf: Path,
    time_window_key: str,
    time_window_label: str = '',
    label_points: bool = True,
    mirror_pdf: Path = None,
) -> None:
    n_panels = max(1, len(conditions))
    fig_width = 6 * n_panels
    fig, axes = plt.subplots(1, n_panels, figsize=(fig_width, 5), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()
    sns.set_style('white')

    for ax, (task_condition, rpe_cat) in zip(axes, conditions):
        panel_df = merged_df[
            (merged_df['task_condition'] == task_condition)
            & (merged_df['rpe_cat'] == rpe_cat)
        ].copy()
        panel_df = panel_df[np.isfinite(panel_df[gc_col]) & np.isfinite(panel_df[target_col])]
        ax.set_title(f'{task_condition} / {rpe_cat}')
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[gc_col].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)
        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, gc_col), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        res_match = results_df[
            (results_df['target_label'] == target_label)
            & (results_df['time_window'] == time_window_key)
            & (results_df['task_condition'] == task_condition)
            & (results_df['rpe_cat'] == rpe_cat)
        ]
        if len(res_match) > 0:
            rr = res_match.iloc[0]
            ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
            ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

        ax.set_xlabel(gc_col)
        ax.set_ylabel(target_col)
        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    tw = time_window_label or time_window_key
    fig.suptitle(f'{target_label} vs {gc_col} ({tw})', fontsize=14)
    _save_and_show_figure(fig, output_pdf, mirror_pdf=mirror_pdf)


def run_predictor_correlations(
    merged_df: pd.DataFrame,
    target_col: str,
    predictor_order: List[str],
    predictor_labels: Dict[str, str],
    analysis_id: str,
    metric_name: str,
) -> pd.DataFrame:
    rows = []
    for predictor in predictor_order:
        stats = run_correlation_analysis(
            x=merged_df[predictor].to_numpy(dtype=float),
            y=merged_df[target_col].to_numpy(dtype=float),
            corr_type=corr_type,
            run_secondary_pearson=run_secondary_pearson,
        )
        rows.append({
            'analysis_id': analysis_id,
            'predictor': predictor,
            'predictor_label': predictor_labels[predictor],
            'target_col': target_col,
            'n': stats['n'],
            'spearman_rho': stats['spearman_rho'],
            'spearman_p': stats['spearman_p'],
            'pearson_r': stats['pearson_r'],
            'pearson_p': stats['pearson_p'],
            'primary_corr_type': stats['primary_corr_type'],
            'primary_r': stats['primary_r'],
            'primary_p': stats['primary_p'],
            'metric_name': metric_name,
        })
    out = pd.DataFrame(rows)
    out['predictor'] = pd.Categorical(out['predictor'], categories=predictor_order, ordered=True)
    return out.sort_values('predictor').reset_index(drop=True)

In [ ]:
# ---- Load subject-level decoding / reinstatement tables ----
for path, label in [
    (acc_reinstatement_csv, 'ACC reinstatement'),
    (acc_hit_decoding_csv, 'ACC hit decoding'),
    (hpc_theta_decoding_csv, 'HPC theta decoding'),
    (acc_hpc_mi_csv, 'ACC-HPC MI'),
]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'{label} CSV not found: {path}')

acc_tbl = pd.read_csv(acc_reinstatement_csv)
acc_hit_tbl = pd.read_csv(acc_hit_decoding_csv)
hpc_theta_tbl = pd.read_csv(hpc_theta_decoding_csv)
acc_hpc_mi_tbl = pd.read_csv(acc_hpc_mi_csv)

for tbl, name in [
    (acc_tbl, 'ACC reinstatement'),
    (acc_hit_tbl, 'ACC hit decoding'),
    (hpc_theta_tbl, 'HPC theta decoding'),
    (acc_hpc_mi_tbl, 'ACC-HPC MI'),
]:
    assert_required_columns(tbl, ['Subject', reinstatement_metric, 'NumElectrodes'], name)

reinstatement_tbl = (
    acc_tbl.rename(columns={
        reinstatement_metric: acc_metric_col,
        'NumElectrodes': 'ACC_NumElectrodes',
    })[['Subject', acc_metric_col, 'ACC_NumElectrodes']]
    .sort_values('Subject')
    .reset_index(drop=True)
)

acc_hit_tbl = (
    acc_hit_tbl.rename(columns={
        'MeanAccuracy': acc_hit_metric_col,
        'NumElectrodes': 'ACC_hit_NumElectrodes',
    })[['Subject', acc_hit_metric_col, 'ACC_hit_NumElectrodes']]
    .sort_values('Subject')
    .reset_index(drop=True)
)

hpc_theta_tbl = (
    hpc_theta_tbl.rename(columns={
        'MeanAccuracy': hpc_theta_metric_col,
        'NumElectrodes': 'HPC_theta_NumElectrodes',
    })[['Subject', hpc_theta_metric_col, 'HPC_theta_NumElectrodes']]
    .sort_values('Subject')
    .reset_index(drop=True)
)

acc_hpc_mi_tbl = (
    acc_hpc_mi_tbl.rename(columns={
        'MeanAccuracy': mi_metric_col,
        'NumElectrodes': 'ACC_HPC_MI_NumElectrodes',
    })[['Subject', mi_metric_col, 'ACC_HPC_MI_NumElectrodes']]
    .sort_values('Subject')
    .reset_index(drop=True)
)

print('Subjects in ACC reinstatement table:', reinstatement_tbl['Subject'].nunique())
print('Subjects in ACC hit decoding table:', acc_hit_tbl['Subject'].nunique())
print('Subjects in HPC theta decoding table:', hpc_theta_tbl['Subject'].nunique())
print('Subjects in ACC-HPC MI table:', acc_hpc_mi_tbl['Subject'].nunique())

behavior_zrpe_hits_tbl = build_zrpe_hits_ols_table(
    combined_csv=behavior_combined_csv,
    min_trials=min_trials_zrpe_hits_ols,
    condition=behavior_day1_condition,
)
behavior_zrpe_hits_tbl.to_csv(paths['behavior_zrpe_hits_csv'], index=False)
print('Subjects in behavior hits~zrpe OLS table:', behavior_zrpe_hits_tbl['Subject'].nunique())
display(behavior_zrpe_hits_tbl.head())

display(reinstatement_tbl.head())

In [ ]:
# ---- Analysis F: ACC reactivation vs HPC theta hit decoding (ACC + HPC subjects) ----
merged_f = deterministic_inner_join(reinstatement_tbl, hpc_theta_tbl, keys=['Subject'])
if len(merged_f) == 0:
    raise ValueError('No overlapping subjects with both ACC and HPC for analysis F.')
if merged_f['Subject'].nunique() < 3:
    raise ValueError('Analysis F requires at least 3 subjects.')

print('Subjects with ACC + HPC (analysis F):', merged_f['Subject'].nunique())
display(
    merged_f[['Subject', acc_metric_col, hpc_theta_metric_col, 'ACC_NumElectrodes', 'HPC_theta_NumElectrodes']]
    .head()
)

results_f = run_predictor_correlations(
    merged_df=merged_f,
    target_col=hpc_theta_metric_col,
    predictor_order=analysis_f_predictor_order,
    predictor_labels=analysis_f_predictor_labels,
    analysis_id='F',
    metric_name='MeanAccuracy',
)

merged_f.to_csv(paths['f_merged_csv'], index=False)
results_f.to_csv(paths['f_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_f,
    results_df=results_f,
    target_col=hpc_theta_metric_col,
    predictor_order=analysis_f_predictor_order,
    predictor_labels=analysis_f_predictor_labels,
    output_pdf=paths['f_scatter_pdf'],
    suptitle=f'{hpc_theta_metric_col} vs ACC reactivation',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['f_scatter_pdf'],
)

print(f'Analysis F: n={len(merged_f)} subjects')
display(results_f)

In [ ]:
# ---- Analysis I: ACC hit decoding vs HPC theta hit decoding ----
merged_i = deterministic_inner_join(acc_hit_tbl, hpc_theta_tbl, keys=['Subject'])
if len(merged_i) == 0:
    raise ValueError('No overlapping subjects for analysis I.')
if merged_i['Subject'].nunique() < 3:
    raise ValueError('Analysis I requires at least 3 subjects.')

print('Subjects with ACC hit + HPC theta (analysis I):', merged_i['Subject'].nunique())
display(
    merged_i[['Subject', acc_hit_metric_col, hpc_theta_metric_col, 'ACC_hit_NumElectrodes', 'HPC_theta_NumElectrodes']]
    .head()
)

results_i = run_predictor_correlations(
    merged_df=merged_i,
    target_col=hpc_theta_metric_col,
    predictor_order=analysis_i_predictor_order,
    predictor_labels=analysis_i_predictor_labels,
    analysis_id='I',
    metric_name='MeanAccuracy',
)

merged_i.to_csv(paths['i_merged_csv'], index=False)
results_i.to_csv(paths['i_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_i,
    results_df=results_i,
    target_col=hpc_theta_metric_col,
    predictor_order=analysis_i_predictor_order,
    predictor_labels=analysis_i_predictor_labels,
    output_pdf=paths['i_scatter_pdf'],
    suptitle=f'{hpc_theta_metric_col} vs ACC cue hit decoding',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['i_scatter_pdf'],
)

print(f'Analysis I: n={len(merged_i)} subjects')
display(results_i)

In [ ]:
# ---- Analysis J: ACC RPE reinstatement vs hits ~ zrpe OLS beta (Day1) ----
merged_j = deterministic_inner_join(reinstatement_tbl, behavior_zrpe_hits_tbl, keys=['Subject'])
if len(merged_j) == 0:
    raise ValueError('No overlapping subjects for analysis J.')
if merged_j['Subject'].nunique() < 3:
    raise ValueError('Analysis J requires at least 3 subjects.')

print('Subjects in ACC reinstatement + hits~zrpe OLS merge (analysis J):', merged_j['Subject'].nunique())
display(merged_j[['Subject', acc_metric_col, zrpe_hits_metric_col, 'n_trials']].head())

results_j = run_predictor_correlations(
    merged_df=merged_j,
    target_col=acc_metric_col,
    predictor_order=analysis_j_predictor_order,
    predictor_labels=analysis_j_predictor_labels,
    analysis_id='J',
    metric_name='zrpe_hits_ols',
)

merged_j.to_csv(paths['j_merged_csv'], index=False)
results_j.to_csv(paths['j_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_j,
    results_df=results_j,
    target_col=acc_metric_col,
    predictor_order=analysis_j_predictor_order,
    predictor_labels=analysis_j_predictor_labels,
    output_pdf=paths['j_scatter_pdf'],
    suptitle=f'{acc_metric_col} vs hits~zrpe OLS beta',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['j_scatter_pdf'],
)

print(f'Analysis J: n={len(merged_j)} subjects')
display(results_j)

In [ ]:
# ---- Analysis A: ACC RPE reinstatement vs ACC hit decoding ----
merged_a = deterministic_inner_join(reinstatement_tbl, acc_hit_tbl, keys=['Subject'])
if len(merged_a) == 0:
    raise ValueError('No overlapping subjects for analysis A.')

results_a = run_predictor_correlations(
    merged_df=merged_a,
    target_col=acc_metric_col,
    predictor_order=analysis_a_predictor_order,
    predictor_labels=analysis_a_predictor_labels,
    analysis_id='A',
    metric_name=reinstatement_metric,
)

merged_a.to_csv(paths['a_merged_csv'], index=False)
results_a.to_csv(paths['a_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_a,
    results_df=results_a,
    target_col=acc_metric_col,
    predictor_order=analysis_a_predictor_order,
    predictor_labels=analysis_a_predictor_labels,
    output_pdf=paths['a_scatter_pdf'],
    suptitle=f'{acc_metric_col} vs ACC cue hit decoding',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['a_scatter_pdf'],
)

print(f'Analysis A: n={len(merged_a)} subjects')
display(results_a)

In [ ]:
# ---- Load GC and build cue_full HPC→ACC table ----
if not os.path.isfile(gc_merged_csv):
    raise FileNotFoundError(f'Merged GC CSV not found: {gc_merged_csv}')

gc_merged_df = pd.read_csv(gc_merged_csv)
assert_required_columns(
    gc_merged_df,
    ['participant', 'roi_source', 'roi_target', 'rpe_cat', 'time', gc_metric],
    'Merged GC table',
)

window_key = 'cue_full'
tmin, tmax = time_windows[window_key]
gc_subject_condition = average_gc_in_time_window(
    gc_merged_df, tmin, tmax, gc_roi_source, gc_roi_target, gc_metric,
)

observed_conditions = set(
    tuple(x)
    for x in gc_subject_condition[['task_condition', 'rpe_cat']].drop_duplicates().itertuples(index=False, name=None)
)
missing_conditions = set(gc_conditions) - observed_conditions
if missing_conditions:
    raise ValueError(f'Missing expected GC condition pairs: {sorted(missing_conditions)}')

gc_pos = gc_subject_condition[gc_subject_condition['rpe_cat'] == 'pos'].copy()
print('GC rows (pos, cue_full):', len(gc_pos))
print('GC subjects (pos):', gc_pos['Subject'].nunique())

In [ ]:
# ---- Analysis B: ACC cue hit decoding vs HPC→ACC GC (pos, cue_full) ----
merged_b = deterministic_inner_join(gc_pos, acc_hit_tbl, keys=['Subject'])
merged_b = merged_b.assign(time_window=window_key)
if len(merged_b) == 0:
    raise ValueError('No overlapping subjects for analysis B.')
if merged_b['Subject'].nunique() < 3:
    raise ValueError('Analysis B requires at least 3 subjects.')

result_rows_b = []
for task_condition, rpe_cat in gc_conditions:
    sub = merged_b[
        (merged_b['task_condition'] == task_condition) & (merged_b['rpe_cat'] == rpe_cat)
    ].copy()
    stats = run_correlation_analysis(
        x=sub[gc_metric].to_numpy(dtype=float),
        y=sub[acc_hit_metric_col].to_numpy(dtype=float),
        corr_type=corr_type,
        run_secondary_pearson=run_secondary_pearson,
    )
    result_rows_b.append({
        'analysis_id': 'B',
        'time_window': window_key,
        'target_label': acc_hit_target_label,
        'target_col': acc_hit_metric_col,
        'task_condition': task_condition,
        'rpe_cat': rpe_cat,
        'gc_metric': gc_metric,
        'n': stats['n'],
        'spearman_rho': stats['spearman_rho'],
        'spearman_p': stats['spearman_p'],
        'pearson_r': stats['pearson_r'],
        'pearson_p': stats['pearson_p'],
        'primary_corr_type': stats['primary_corr_type'],
        'primary_r': stats['primary_r'],
        'primary_p': stats['primary_p'],
    })

results_b = pd.DataFrame(result_rows_b)
merged_b.to_csv(paths['b_merged_csv'], index=False)
results_b.to_csv(paths['b_results_csv'], index=False)

plot_roi_condition_grid(
    merged_df=merged_b,
    results_df=results_b,
    target_col=acc_hit_metric_col,
    target_label=acc_hit_target_label,
    gc_col=gc_metric,
    conditions=gc_conditions,
    output_pdf=paths['b_scatter_pdf'],
    time_window_key=window_key,
    time_window_label=time_window_labels[window_key],
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['b_scatter_pdf'],
)

print(f'Analysis B: n={merged_b["Subject"].nunique()} subjects')
display(results_b)

In [ ]:
# ---- RSA cluster reconstruction (ACC→HPC hit RSA) ----
if not os.path.isfile(rsa_group_results_pkl):
    raise FileNotFoundError(f'RSA group results PKL not found: {rsa_group_results_pkl}')

with open(rsa_group_results_pkl, 'rb') as f:
    rsa_group_results = pickle.load(f)

required_rsa_keys = [
    'observed_diffs_stack',
    'feedback_times',
    'cue_times',
    'subject_results',
    'successful_subjects',
    'n_subjects',
]
missing_rsa_keys = [k for k in required_rsa_keys if k not in rsa_group_results]
if missing_rsa_keys:
    raise ValueError(f'RSA group results missing required keys: {missing_rsa_keys}')

X = np.asarray(rsa_group_results['observed_diffs_stack'], dtype=float)
feedback_times = np.asarray(rsa_group_results['feedback_times'], dtype=float)
cue_times = np.asarray(rsa_group_results['cue_times'], dtype=float)

if X.ndim != 3:
    raise ValueError(f'Expected observed_diffs_stack to be 3D; got shape {X.shape}.')

n_fb = X.shape[1]
n_cue = X.shape[2]
fb_adjacency = mne.stats.combine_adjacency(n_fb)
cue_adjacency = mne.stats.combine_adjacency(n_cue)
adjacency_2d = csr_matrix(kron(eye(n_fb), cue_adjacency) + kron(fb_adjacency, eye(n_cue)))
X_flat = X.reshape(X.shape[0], -1)

t_obs, clusters, cluster_pv, H0 = permutation_cluster_1samp_test(
    X_flat,
    n_permutations=cluster_n_permutations,
    threshold=None,
    adjacency=adjacency_2d,
    tail=cluster_tail,
    n_jobs=8,
    seed=cluster_seed,
    verbose=False,
)

sig_cluster_indices = [i for i, pval in enumerate(cluster_pv) if pval < cluster_alpha]
if len(sig_cluster_indices) != 1:
    raise ValueError(
        f'Expected exactly 1 significant ACC→HPC cluster at alpha={cluster_alpha}; '
        f'found {len(sig_cluster_indices)}.'
    )

sig_cluster_index = sig_cluster_indices[0]
sig_cluster = clusters[sig_cluster_index]
sig_mask_flat = np.zeros(n_fb * n_cue, dtype=bool)
sig_mask_flat[sig_cluster[0]] = True
sig_cluster_mask = sig_mask_flat.reshape(n_fb, n_cue)
sig_cluster_size = int(sig_cluster_mask.sum())

fb_indices = np.where(sig_cluster_mask.any(axis=1))[0]
cue_indices = np.where(sig_cluster_mask.any(axis=0))[0]
if len(fb_indices) == 0 or len(cue_indices) == 0:
    raise ValueError('Significant cluster mask is empty after reshaping.')

cluster_metadata = {
    'feedback_roi': rsa_feedback_roi,
    'cue_roi': rsa_cue_roi,
    'cluster_index_zero_based': int(sig_cluster_index),
    'cluster_p_value': float(cluster_pv[sig_cluster_index]),
    'cluster_size': sig_cluster_size,
    'n_cluster_subjects': int(X.shape[0]),
    'n_feedback_timepoints': int(n_fb),
    'n_cue_timepoints': int(n_cue),
    'feedback_tmin': float(feedback_times[fb_indices[0]]),
    'feedback_tmax': float(feedback_times[fb_indices[-1]]),
    'cue_tmin': float(cue_times[cue_indices[0]]),
    'cue_tmax': float(cue_times[cue_indices[-1]]),
    'cluster_alpha': float(cluster_alpha),
    'cluster_n_permutations': int(cluster_n_permutations),
    'cluster_tail': int(cluster_tail),
    'cluster_seed': int(cluster_seed),
}
cluster_metadata_df = pd.DataFrame([cluster_metadata])
cluster_metadata_df.to_csv(paths['cluster_metadata_csv'], index=False)

rsa_rows = []
for subject_result in rsa_group_results['subject_results']:
    observed = subject_result['observed']
    subject = observed['subject']
    category_heatmaps = observed.get('category_heatmaps', {})
    if hit_memory_label not in category_heatmaps:
        raise ValueError(f'Subject {subject} missing category_heatmaps[{hit_memory_label!r}].')
    hit_heatmap = np.asarray(category_heatmaps[hit_memory_label], dtype=float)
    if hit_heatmap.shape != sig_cluster_mask.shape:
        raise ValueError(
            f'Subject {subject} heatmap shape {hit_heatmap.shape} does not match cluster mask.'
        )
    rsa_rows.append({
        'Subject': subject,
        rsa_metric_col: float(np.nanmean(hit_heatmap[sig_cluster_mask])),
    })

rsa_subject_tbl = pd.DataFrame(rsa_rows).sort_values('Subject').reset_index(drop=True)
assert_required_columns(rsa_subject_tbl, ['Subject', rsa_metric_col], 'RSA subject table')

print('ACC→HPC cluster p =', cluster_metadata['cluster_p_value'])
print('Subjects in RSA table:', rsa_subject_tbl['Subject'].nunique())
display(cluster_metadata_df)

In [ ]:
# ---- Analysis E: ACC cue hit decoding vs ACC→HPC hit RSA ----
merged_e = deterministic_inner_join(acc_hit_tbl, rsa_subject_tbl, keys=['Subject'])
if len(merged_e) == 0:
    raise ValueError('No overlapping subjects for analysis E.')

results_e = run_predictor_correlations(
    merged_df=merged_e,
    target_col=acc_hit_metric_col,
    predictor_order=analysis_e_predictor_order,
    predictor_labels=analysis_e_predictor_labels,
    analysis_id='E',
    metric_name='MeanAccuracy',
)

merged_e.to_csv(paths['e_merged_csv'], index=False)
results_e.to_csv(paths['e_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_e,
    results_df=results_e,
    target_col=acc_hit_metric_col,
    predictor_order=analysis_e_predictor_order,
    predictor_labels=analysis_e_predictor_labels,
    output_pdf=paths['e_scatter_pdf'],
    suptitle=f'{acc_hit_metric_col} vs ACC→HPC hit RSA',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['e_scatter_pdf'],
)

print(f'Analysis E: n={len(merged_e)} subjects')
display(results_e)

In [ ]:
# ---- Analysis K: ACC RPE crossdecoding vs ACC→HPC hit RSA ----
merged_k = deterministic_inner_join(reinstatement_tbl, rsa_subject_tbl, keys=['Subject'])
if len(merged_k) == 0:
    raise ValueError('No overlapping subjects for analysis K.')
if merged_k['Subject'].nunique() < 3:
    raise ValueError('Analysis K requires at least 3 subjects.')

print('Subjects in ACC crossdecoding + hit RSA merge (analysis K):', merged_k['Subject'].nunique())
display(merged_k[['Subject', acc_metric_col, rsa_metric_col]].head())

results_k = run_predictor_correlations(
    merged_df=merged_k,
    target_col=rsa_metric_col,
    predictor_order=analysis_k_predictor_order,
    predictor_labels=analysis_k_predictor_labels,
    analysis_id='K',
    metric_name='HitRSA_cluster_mean',
)

merged_k.to_csv(paths['k_merged_csv'], index=False)
results_k.to_csv(paths['k_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_k,
    results_df=results_k,
    target_col=rsa_metric_col,
    predictor_order=analysis_k_predictor_order,
    predictor_labels=analysis_k_predictor_labels,
    output_pdf=paths['k_scatter_pdf'],
    suptitle=f'{rsa_metric_col} vs ACC RPE crossdecoding',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['k_scatter_pdf'],
)

print(f'Analysis K: n={len(merged_k)} subjects')
display(results_k)

In [ ]:
# ---- Analysis G: HPC→ACC GC vs ACC→HPC hit RSA (pos, cue_full) ----
merged_g = deterministic_inner_join(gc_pos, rsa_subject_tbl, keys=['Subject'])
if len(merged_g) == 0:
    raise ValueError('No overlapping subjects for analysis G.')
if merged_g['Subject'].nunique() < 3:
    raise ValueError('Analysis G requires at least 3 subjects.')

print('Subjects in GC + RSA merged table (analysis G):', merged_g['Subject'].nunique())
display(merged_g[['Subject', gc_metric, rsa_metric_col]].head())

results_g = run_predictor_correlations(
    merged_df=merged_g,
    target_col=rsa_metric_col,
    predictor_order=analysis_g_predictor_order,
    predictor_labels=analysis_g_predictor_labels,
    analysis_id='G',
    metric_name=rsa_metric_name,
)

merged_g.to_csv(paths['g_merged_csv'], index=False)
results_g.to_csv(paths['g_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_g,
    results_df=results_g,
    target_col=rsa_metric_col,
    predictor_order=analysis_g_predictor_order,
    predictor_labels=analysis_g_predictor_labels,
    output_pdf=paths['g_scatter_pdf'],
    suptitle=f'{rsa_metric_col} vs HPC→ACC theta GC',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['g_scatter_pdf'],
)

print(f'Analysis G: n={len(merged_g)} subjects')
display(results_g)

In [ ]:
# ---- Analysis H: ACC↔HPC MI cluster effect vs decoding / GC (pos, cue_full) ----
merged_h = deterministic_inner_join(acc_hpc_mi_tbl, reinstatement_tbl, keys=['Subject'])
merged_h = deterministic_inner_join(merged_h, acc_hit_tbl, keys=['Subject'])
merged_h = deterministic_inner_join(merged_h, hpc_theta_tbl, keys=['Subject'])
merged_h = deterministic_inner_join(
    merged_h,
    gc_pos[['Subject', gc_metric, 'task_condition', 'rpe_cat']],
    keys=['Subject'],
)
if len(merged_h) == 0:
    raise ValueError('No overlapping subjects for analysis H.')
if merged_h['Subject'].nunique() < 3:
    raise ValueError('Analysis H requires at least 3 subjects.')

print('Subjects in MI + decoding/GC merged table (analysis H):', merged_h['Subject'].nunique())
display(merged_h[['Subject', mi_metric_col] + analysis_h_predictor_order].head())

results_h = run_predictor_correlations(
    merged_df=merged_h,
    target_col=mi_metric_col,
    predictor_order=analysis_h_predictor_order,
    predictor_labels=analysis_h_predictor_labels,
    analysis_id='H',
    metric_name='MI_cluster_effect',
)

merged_h.to_csv(paths['h_merged_csv'], index=False)
results_h.to_csv(paths['h_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_h,
    results_df=results_h,
    target_col=mi_metric_col,
    predictor_order=analysis_h_predictor_order,
    predictor_labels=analysis_h_predictor_labels,
    output_pdf=paths['h_scatter_pdf'],
    suptitle='ACC↔HPC multivariate MI (cluster effect) vs decoding / GC',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['h_scatter_pdf'],
)

print(f'Analysis H: n={len(merged_h)} subjects')
display(results_h)

In [ ]:
# ---- Analysis C: HPC theta decoding vs ACC→HPC hit RSA ----
merged_c = deterministic_inner_join(hpc_theta_tbl, rsa_subject_tbl, keys=['Subject'])
if len(merged_c) == 0:
    raise ValueError('No overlapping subjects for analysis C.')

results_c = run_predictor_correlations(
    merged_df=merged_c,
    target_col=hpc_theta_metric_col,
    predictor_order=analysis_c_predictor_order,
    predictor_labels=analysis_c_predictor_labels,
    analysis_id='C',
    metric_name='MeanAccuracy',
)

merged_c.to_csv(paths['c_merged_csv'], index=False)
results_c.to_csv(paths['c_results_csv'], index=False)
plot_subject_covariate_grid(
    merged_df=merged_c,
    results_df=results_c,
    target_col=hpc_theta_metric_col,
    predictor_order=analysis_c_predictor_order,
    predictor_labels=analysis_c_predictor_labels,
    output_pdf=paths['c_scatter_pdf'],
    suptitle=f'{hpc_theta_metric_col} vs ACC→HPC hit RSA',
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['c_scatter_pdf'],
)

print(f'Analysis C: n={len(merged_c)} subjects')
display(results_c)

In [ ]:
# ---- Analysis D: HPC theta decoding vs HPC→ACC GC (pos, cue_full) ----
merged_d = deterministic_inner_join(gc_pos, hpc_theta_tbl, keys=['Subject'])
merged_d = merged_d.assign(time_window=window_key)
if len(merged_d) == 0:
    raise ValueError('No overlapping subjects for analysis D.')
if merged_d['Subject'].nunique() < 3:
    raise ValueError('Analysis D requires at least 3 subjects.')

result_rows_d = []
for task_condition, rpe_cat in gc_conditions:
    sub = merged_d[
        (merged_d['task_condition'] == task_condition) & (merged_d['rpe_cat'] == rpe_cat)
    ].copy()
    stats = run_correlation_analysis(
        x=sub[gc_metric].to_numpy(dtype=float),
        y=sub[hpc_theta_metric_col].to_numpy(dtype=float),
        corr_type=corr_type,
        run_secondary_pearson=run_secondary_pearson,
    )
    result_rows_d.append({
        'analysis_id': 'D',
        'time_window': window_key,
        'target_label': hpc_target_label,
        'target_col': hpc_theta_metric_col,
        'task_condition': task_condition,
        'rpe_cat': rpe_cat,
        'gc_metric': gc_metric,
        'n': stats['n'],
        'spearman_rho': stats['spearman_rho'],
        'spearman_p': stats['spearman_p'],
        'pearson_r': stats['pearson_r'],
        'pearson_p': stats['pearson_p'],
        'primary_corr_type': stats['primary_corr_type'],
        'primary_r': stats['primary_r'],
        'primary_p': stats['primary_p'],
    })

results_d = pd.DataFrame(result_rows_d)
merged_d.to_csv(paths['d_merged_csv'], index=False)
results_d.to_csv(paths['d_results_csv'], index=False)

plot_roi_condition_grid(
    merged_df=merged_d,
    results_df=results_d,
    target_col=hpc_theta_metric_col,
    target_label=hpc_target_label,
    gc_col=gc_metric,
    conditions=gc_conditions,
    output_pdf=paths['d_scatter_pdf'],
    time_window_key=window_key,
    time_window_label=time_window_labels[window_key],
    label_points=label_points,
    mirror_pdf=manuscript_figure_paths['d_scatter_pdf'],
)

print(f'Analysis D: n={merged_d["Subject"].nunique()} subjects')
display(results_d)

In [ ]:
# ---- Summary ----
summary_lines = [
    'ACC/HPC decoding, GC, and RSA subject-level correlations',
    '=' * 60,
    f'Primary correlation: {corr_type} (secondary Pearson: {run_secondary_pearson})',
    f'GC pair: {gc_roi_source} -> {gc_roi_target} ({freq_band_label}, cue_full 0-1 s, pos RPE)',
    f'RSA cluster p-value: {cluster_metadata["cluster_p_value"]:.6f}',
    '',
    'Data sources:',
    f'  ACC reinstatement: {acc_reinstatement_csv}',
    f'  ACC hit decoding: {acc_hit_decoding_csv}',
    f'  HPC theta decoding: {hpc_theta_decoding_csv}',
    f'  GC merged: {gc_merged_csv}',
    f'  RSA group PKL: {rsa_group_results_pkl}',
    f'  ACC-HPC MI: {acc_hpc_mi_csv}',
    f'  Behavior combined: {behavior_combined_csv}',
    f'  Behavior hits~zrpe OLS audit: {paths["behavior_zrpe_hits_csv"]}',
    f'  hits~zrpe OLS: Day1 trials, min_trials={min_trials_zrpe_hits_ols}',
    '',
    f'Analysis A subjects: {merged_a["Subject"].nunique()}',
    f'Analysis B subjects: {merged_b["Subject"].nunique()}',
    f'Analysis C subjects: {merged_c["Subject"].nunique()}',
    f'Analysis D subjects: {merged_d["Subject"].nunique()}',
    f'Analysis E subjects: {merged_e["Subject"].nunique()}',
    f'Analysis F subjects: {merged_f["Subject"].nunique()}',
    f'Analysis G subjects: {merged_g["Subject"].nunique()}',
    f'Analysis H subjects: {merged_h["Subject"].nunique()}',
    f'Analysis I subjects: {merged_i["Subject"].nunique()}',
    f'Analysis J subjects: {merged_j["Subject"].nunique()}',
    f'Analysis K subjects: {merged_k["Subject"].nunique()}',
    '',
    'Analysis A (ACC reinstatement vs ACC hit decoding):',
]
for row in results_a.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis B (ACC hit decoding vs GC):')
for row in results_b.itertuples(index=False):
    summary_lines.append(
        f'  {row.task_condition}/{row.rpe_cat}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis C (HPC theta vs hit RSA):')
for row in results_c.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis D (HPC theta vs GC):')
for row in results_d.itertuples(index=False):
    summary_lines.append(
        f'  {row.task_condition}/{row.rpe_cat}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis E (ACC hit decoding vs hit RSA):')
for row in results_e.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis K (ACC crossdecoding vs ACC→HPC hit RSA):')
for row in results_k.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis F (ACC reactivation vs HPC theta hit decoding):')
for row in results_f.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis I (ACC hit decoding vs HPC theta hit decoding):')
for row in results_i.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis J (ACC reinstatement vs hits ~ zrpe OLS beta):')
for row in results_j.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis G (HPC→ACC GC vs ACC→HPC hit RSA):')
for row in results_g.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
summary_lines.append('')
summary_lines.append('Analysis H (ACC↔HPC MI cluster effect vs decoding / GC):')
for row in results_h.itertuples(index=False):
    summary_lines.append(
        f'  {row.predictor_label}: {row.primary_corr_type} r={row.primary_r:.4f}, '
        f'p={row.primary_p:.4g}, n={row.n}'
    )
if os.path.isfile(acc_hpc_mi_metadata_csv):
    mi_meta = pd.read_csv(acc_hpc_mi_metadata_csv)
    if len(mi_meta) > 0:
        row0 = mi_meta.iloc[0]
        p_col = 'p_value' if 'p_value' in mi_meta.columns else 'cluster_p_value'
        t0_col = 'start_time_s' if 'start_time_s' in mi_meta.columns else None
        t1_col = 'end_time_s' if 'end_time_s' in mi_meta.columns else None
        meta_bits = [f'p={row0[p_col]:.4f}' if p_col in mi_meta.columns else None]
        if t0_col and t1_col and t0_col in mi_meta.columns and t1_col in mi_meta.columns:
            meta_bits.append(f'window={row0[t0_col]:.3f}-{row0[t1_col]:.3f} s')
        summary_lines.append('  MI cluster metadata: ' + ', '.join(b for b in meta_bits if b))
summary_lines.append('')
summary_lines.append('Figure outputs (PDF + PNG in repo):')
for key in ['a_scatter_pdf', 'b_scatter_pdf', 'c_scatter_pdf', 'd_scatter_pdf', 'e_scatter_pdf', 'f_scatter_pdf', 'g_scatter_pdf', 'h_scatter_pdf', 'i_scatter_pdf', 'j_scatter_pdf', 'k_scatter_pdf']:
    summary_lines.append(f'  {key}: {paths[key]}')
    summary_lines.append(f'  {key}_png: {paths[key].with_suffix(".png")}')
summary_lines.append('')
summary_lines.append('Table outputs:')
for key, path in sorted(paths.items()):
    if 'scatter' not in key:
        summary_lines.append(f'  {key}: {path}')

summary_text = '\n'.join(summary_lines)
paths['summary_txt'].write_text(summary_text)
print(summary_text)
print('\nSaved summary:', paths['summary_txt'])